In [ ]:
# TURMA 10DTSR
# LUCAS SOUSA: 358447
# BRUNO GOMES: 358853

# https://youtu.be/bbfRzDx9UDc

In [5]:
!pip install opencv-python dlib tensorflow keras numpy matplotlib
!wget http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
!bunzip2 shape_predictor_68_face_landmarks.dat.bz2

print("✅ Dependências instaladas com sucesso!")

import cv2
import dlib
import numpy as np
import tensorflow as tf
from tensorflow import keras
from keras import layers
import time
import matplotlib.pyplot as plt
from IPython.display import display, Javascript, Image
from google.colab.output import eval_js
from base64 import b64decode
import io
from PIL import Image as PILImage

print("📚 Bibliotecas importadas com sucesso!")

class FacialAuthenticationSystem:
    def __init__(self):
        print("🔄 Inicializando sistema de autenticação facial...")

        # Inicializar detector de faces do dlib
        self.face_detector = dlib.get_frontal_face_detector()
        print("✅ Detector de faces inicializado")

        # Carregar predictor de landmarks
        try:
            self.landmark_predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
            print("✅ Predictor de landmarks carregado")
        except:
            print("⚠️  Não foi possível carregar o predictor de landmarks")
            self.landmark_predictor = None

        # Simular modelo de vivacidade (em produção seria um modelo treinado)
        self.liveness_threshold = 0.8
        print("✅ Sistema de vivacidade configurado")

        # Banco de dados simulado de usuários
        self.user_database = {
            "user_001": {
                "name": "João Silva",
                "embedding": np.random.randn(128),
                "threshold": 0.7
            },
            "user_002": {
                "name": "Maria Santos",
                "embedding": np.random.randn(128),
                "threshold": 0.7
            }
        }

        # Variáveis para detecção de vivacidade
        self.eye_blink_counter = 0
        self.movement_detected = False
        self.last_head_position = None

        print("🎉 Sistema de autenticação facial inicializado com sucesso!")

    def detect_faces(self, frame):
        """Detecta faces no frame usando dlib"""
        try:
            # Converter BGR para RGB
            rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            faces = self.face_detector(rgb_frame)
            return faces
        except Exception as e:
            print(f"❌ Erro na detecção de faces: {e}")
            return []

    def get_face_landmarks(self, frame, face):
        """Extrai landmarks faciais"""
        if self.landmark_predictor is None:
            return None

        try:
            landmarks = self.landmark_predictor(frame, face)
            return landmarks
        except Exception as e:
            print(f"❌ Erro ao extrair landmarks: {e}")
            return None

    def calculate_eye_aspect_ratio(self, eye_points):
        """Calcula a proporção de abertura dos olhos para detecção de piscadas"""
        # Calcular distâncias verticais
        A = np.linalg.norm(eye_points[1] - eye_points[5])
        B = np.linalg.norm(eye_points[2] - eye_points[4])

        # Calcular distância horizontal
        C = np.linalg.norm(eye_points[0] - eye_points[3])

        # Calcular Eye Aspect Ratio
        ear = (A + B) / (2.0 * C)
        return ear

    def check_eye_blink(self, landmarks):
        """Verifica se há piscada de olhos"""
        if landmarks is None:
            return False

        # Pontos para olho esquerdo (índices 36-41)
        left_eye_points = np.array([(landmarks.part(i).x, landmarks.part(i).y) for i in range(36, 42)])

        # Pontos para olho direito (índices 42-47)
        right_eye_points = np.array([(landmarks.part(i).x, landmarks.part(i).y) for i in range(42, 48)])

        # Calcular EAR para ambos os olhos
        left_ear = self.calculate_eye_aspect_ratio(left_eye_points)
        right_ear = self.calculate_eye_aspect_ratio(right_eye_points)

        # EAR médio
        avg_ear = (left_ear + right_ear) / 2.0

        # Limiar para detecção de piscada
        return avg_ear < 0.2

    def check_head_movement(self, landmarks):
        """Detecta movimento da cabeça"""
        if landmarks is None:
            return False

        # Calcular posição média da cabeça baseada nos landmarks
        nose_point = np.array([landmarks.part(30).x, landmarks.part(30).y])
        left_eye_point = np.array([landmarks.part(36).x, landmarks.part(36).y])
        right_eye_point = np.array([landmarks.part(45).x, landmarks.part(45).y])

        current_position = np.mean([nose_point, left_eye_point, right_eye_point], axis=0)

        if self.last_head_position is not None:
            # Calcular distância do movimento
            movement = np.linalg.norm(current_position - self.last_head_position)
            if movement > 5:  # Limiar para movimento significativo
                self.movement_detected = True

        self.last_head_position = current_position
        return self.movement_detected

    def extract_facial_embedding(self, face_roi):
        """Extrai embedding facial (simulado)"""
        # Em produção, usaria um modelo como FaceNet
        return np.random.randn(128)

    def verify_user(self, embedding):
        """Verifica se o embedding corresponde a algum usuário"""
        best_match = None
        best_score = 0

        for user_id, user_data in self.user_database.items():
            # Calcular similaridade cosseno
            similarity = np.dot(embedding, user_data["embedding"]) / (
                np.linalg.norm(embedding) * np.linalg.norm(user_data["embedding"])
            )

            if similarity > user_data["threshold"] and similarity > best_score:
                best_score = similarity
                best_match = user_id

        return best_match, best_score

    def check_liveness(self, frame, landmarks, frame_count):
        """Verifica vivacidade usando múltiplas técnicas"""
        liveness_score = 0.0
        checks_passed = 0

        # 1. Verificar piscada de olhos
        if frame_count % 30 == 0:  # Verificar a cada 30 frames
            if self.check_eye_blink(landmarks):
                self.eye_blink_counter += 1
                print("👁️  Piscada detectada!")

        if self.eye_blink_counter >= 1:
            liveness_score += 0.4
            checks_passed += 1

        # 2. Verificar movimento da cabeça
        if self.check_head_movement(landmarks):
            liveness_score += 0.4
            checks_passed += 1

        # 3. Verificar qualidade da imagem (simulado)
        # Em produção, analisaria textura, reflexos, etc.
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        blur_value = cv2.Laplacian(gray, cv2.CV_64F).var()
        if blur_value > 100:  # Imagem não muito borrada
            liveness_score += 0.2
            checks_passed += 1

        return liveness_score >= self.liveness_threshold, liveness_score

    def draw_face_info(self, frame, face, landmarks, user_id, liveness_score, authenticated):
        """Desenha informações na tela"""
        # Desenhar retângulo ao redor da face
        x, y, w, h = face.left(), face.top(), face.width(), face.height()
        color = (0, 255, 0) if authenticated else (0, 0, 255)
        cv2.rectangle(frame, (x, y), (x + w, y + h), color, 2)

        # Desenhar landmarks
        if landmarks:
            for i in range(68):
                x_point = landmarks.part(i).x
                y_point = landmarks.part(i).y
                cv2.circle(frame, (x_point, y_point), 2, (255, 0, 0), -1)

        # Adicionar informações de texto
        status = "AUTENTICADO" if authenticated else "NÃO AUTENTICADO"
        user_text = f"Usuário: {user_id}" if user_id else "Usuário: Não identificado"

        cv2.putText(frame, status, (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, color, 2)
        cv2.putText(frame, user_text, (x, y-40), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(frame, f"Vivacidade: {liveness_score:.2f}", (x, y-60), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)

        # Instruções
        cv2.putText(frame, "Pisque os olhos e mova a cabeca", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        cv2.putText(frame, "Pressione 'q' para sair", (10, 60), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

    def authenticate_from_webcam(self):
        """Executa autenticação facial via webcam"""
        print("📷 Iniciando captura da webcam...")
        print("💡 Instruções:")
        print("   - Posicione seu rosto na câmera")
        print("   - Pisque os olhos naturalmente")
        print("   - Faça pequenos movimentos com a cabeça")
        print("   - Pressione 'q' para sair")

        # Inicializar webcam
        cap = cv2.VideoCapture(0)

        if not cap.isOpened():
            print("❌ Não foi possível acessar a webcam")
            return

        frame_count = 0
        authentication_attempts = 0
        max_attempts = 3

        while True:
            ret, frame = cap.read()
            if not ret:
                print("❌ Erro ao capturar frame")
                break

            # Espelhar o frame para efeito espelho
            frame = cv2.flip(frame, 1)

            # Detectar faces
            faces = self.detect_faces(frame)

            authenticated = False
            user_id = None
            liveness_score = 0.0

            if len(faces) > 0:
                # Usar a primeira face detectada
                face = faces[0]

                # Extrair landmarks
                landmarks = self.get_face_landmarks(frame, face)

                # Verificar vivacidade
                liveness_passed, liveness_score = self.check_liveness(frame, landmarks, frame_count)

                if liveness_passed:
                    # Extrair embedding facial
                    embedding = self.extract_facial_embedding(frame)

                    # Verificar usuário
                    user_id, confidence = self.verify_user(embedding)

                    if user_id and confidence > 0.7:
                        authenticated = True
                        user_name = self.user_database[user_id]["name"]
                        print(f"🎉 Autenticação bem-sucedida! Usuário: {user_name}")
                    else:
                        print("❌ Usuário não identificado no banco de dados")
                else:
                    print(f"⚠️  Vivacidade insuficiente: {liveness_score:.2f}")

                # Desenhar informações na tela
                self.draw_face_info(frame, face, landmarks, user_id, liveness_score, authenticated)

            else:
                cv2.putText(frame, "Nenhuma face detectada", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)

            # Mostrar frame
            cv2.imshow('Sistema de Autenticação Facial - Banco', frame)

            # Verificar se autenticou com sucesso
            if authenticated:
                time.sleep(2)  # Mostrar resultado por 2 segundos
                break

            # Contar tentativas
            if len(faces) > 0 and not authenticated:
                authentication_attempts += 1
                if authentication_attempts >= max_attempts:
                    print("🚫 Número máximo de tentativas atingido")
                    cv2.putText(frame, "REDIRECIONANDO PARA ESTEIRA DEDICADA", (50, 150),
                               cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 3)
                    cv2.imshow('Sistema de Autenticação Facial - Banco', frame)
                    cv2.waitKey(3000)
                    break

            # Sair com 'q'
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

            frame_count += 1

        # Liberar recursos
        cap.release()
        cv2.destroyAllWindows()

        # Resultado final
        if authenticated:
            print("✅ AUTENTICAÇÃO BEM-SUCEDIDA - ACESSO PERMITIDO")
            return True
        else:
            print("❌ FALHA NA AUTENTICAÇÃO - ENCAMINHANDO PARA ESTEIRA DEDICADA")
            print("📋 Evidências serão enviadas para área de IA para aperfeiçoamento do modelo")
            return False

# Função para demonstrar o sistema
def demonstrate_authentication_system():
    """Demonstra o sistema completo de autenticação"""
    print("=" * 60)
    print("🏦 SISTEMA DE AUTENTICAÇÃO FACIAL - BANCO DIGITAL")
    print("=" * 60)

    # Inicializar sistema
    auth_system = FacialAuthenticationSystem()

    # Executar autenticação
    print("\n🔄 Iniciando processo de autenticação...")
    success = auth_system.authenticate_from_webcam()

    # Resultado
    if success:
        print("\n🎉 Cliente autenticado com sucesso!")
        print("✅ Acesso permitido aos serviços bancários")
    else:
        print("\n🚫 Cliente não autenticado")
        print("📋 Protocolo de segurança ativado:")
        print("   - Cliente será atendido por esteira dedicada")
        print("   - Evidências encaminhadas para área de IA")
        print("   - Parâmetros serão ajustados para aperfeiçoamento do modelo")

    print("\n" + "=" * 60)
    print("🔒 Sistema de segurança encerrado")
    print("=" * 60)

# Executar demonstração
if __name__ == "__main__":
    demonstrate_authentication_system()

--2025-10-12 21:28:30--  http://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Resolving dlib.net (dlib.net)... 107.180.26.78
Connecting to dlib.net (dlib.net)|107.180.26.78|:80... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2 [following]
--2025-10-12 21:28:30--  https://dlib.net/files/shape_predictor_68_face_landmarks.dat.bz2
Connecting to dlib.net (dlib.net)|107.180.26.78|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64040097 (61M)
Saving to: ‘shape_predictor_68_face_landmarks.dat.bz2’

shape_predictor_68_ 100%[===================>]  61.07M  14.7MB/s    in 5.4s    

2025-10-12 21:28:37 (11.3 MB/s) - ‘shape_predictor_68_face_landmarks.dat.bz2’ saved [64040097/64040097]

bunzip2: Output file shape_predictor_68_face_landmarks.dat already exists.
✅ Dependências instaladas com sucesso!
📚 Bibliotecas importadas com sucesso!
🏦 SISTEMA DE AUTENTICAÇÃO FA

In [7]:
# CÓDIGO ALTERNATIVO PARA COLAB
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode
import numpy as np
import cv2
import PIL

def take_photo(filename='photo.jpg', quality=0.8):
    """Tira foto usando a webcam do Colab"""
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture Photo';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            // Resize the output to fit the video element.
            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            // Wait for Capture to be clicked.
            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
    ''')
    display(js)

    # Tirar foto
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])

    # Salvar imagem
    with open(filename, 'wb') as f:
        f.write(binary)

    return filename

def analyze_photo(filename):
    """Analisa a foto para autenticação facial"""
    print("🔍 Analisando foto para autenticação...")

    # Carregar imagem
    image = cv2.imread(filename)

    # Inicializar detector de faces
    face_detector = dlib.get_frontal_face_detector()

    # Detectar faces
    rgb_image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    faces = face_detector(rgb_image)

    if len(faces) > 0:
        print(f"✅ {len(faces)} face(s) detectada(s)")

        # Desenhar retângulos nas faces
        for face in faces:
            x, y, w, h = face.left(), face.top(), face.width(), face.height()
            cv2.rectangle(image, (x, y), (x + w, y + h), (0, 255, 0), 2)

        # Salvar imagem com detecção
        output_filename = 'detected_' + filename
        cv2.imwrite(output_filename, image)

        print("📊 Simulando verificação de vivacidade...")
        print("✅ Movimento detectado: Simulado")
        print("✅ Piscada de olhos: Simulada")
        print("✅ Textura facial: Autêntica")

        return True, output_filename
    else:
        print("❌ Nenhuma face detectada")
        return False, filename

# Demonstração simplificada
def simple_demo():
    print("🎯 DEMONSTRAÇÃO SIMPLIFICADA - SISTEMA DE AUTENTICAÇÃO FACIAL")
    print("=" * 50)

    # Tirar foto
    print("\n1. 📸 Capturando imagem da webcam...")
    photo_file = take_photo()

    # Analisar foto
    print("\n2. 🔍 Processando autenticação facial...")
    authenticated, result_file = analyze_photo(photo_file)

    # Resultado
    print("\n3. 📋 RESULTADO DA AUTENTICAÇÃO:")
    if authenticated:
        print("   ✅ AUTENTICADO - Acesso permitido")
        print("   🎉 Cliente identificado com sucesso")
    else:
        print("   ❌ NÃO AUTENTICADO - Encaminhando para esteira dedicada")
        print("   📨 Evidências enviadas para área de IA")

    # Mostrar resultado
    print(f"\n4. 🖼️  Imagem processada salva como: {result_file}")

# Executar demonstração simplificada
simple_demo()

🎯 DEMONSTRAÇÃO SIMPLIFICADA - SISTEMA DE AUTENTICAÇÃO FACIAL

1. 📸 Capturando imagem da webcam...


<IPython.core.display.Javascript object>


2. 🔍 Processando autenticação facial...
🔍 Analisando foto para autenticação...
✅ 1 face(s) detectada(s)
📊 Simulando verificação de vivacidade...
✅ Movimento detectado: Simulado
✅ Piscada de olhos: Simulada
✅ Textura facial: Autêntica

3. 📋 RESULTADO DA AUTENTICAÇÃO:
   ✅ AUTENTICADO - Acesso permitido
   🎉 Cliente identificado com sucesso

4. 🖼️  Imagem processada salva como: detected_photo.jpg
